# 02 — Calidad de datos

Acá documento **qué tiré y por qué**. Es la notebook que más se mira en una
entrevista técnica: muestra criterio, no código.

Regla que sigo: nunca descarto sin medir el impacto. Si una regla me borra más
del 5% de la base, la regla está mal, no los datos.

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import duckdb, pandas as pd, matplotlib.pyplot as plt
from src import config as cfg
from src.canasta import cobertura, etiquetar, canasta_df

pd.set_option('display.max_columns', 50)
plt.rcParams.update({'figure.figsize': (11, 5), 'axes.grid': True, 'grid.alpha': .3})
con = duckdb.connect()
PARQUET = str(cfg.INTERIM / 'fecha=*' / '*.parquet')


### 1. Resumen de descartes del pipeline

In [ ]:
cal = pd.read_csv(cfg.INTERIM / '_calidad.csv')
display(cal)

desc = [c for c in cal.columns if c.startswith('desc_')]
ax = (100 * cal[desc].sum() / cal.filas_crudas.sum()).sort_values().plot.barh()
ax.set_title('% de filas descartadas por motivo'); ax.set_xlabel('% del total crudo');

### 2. Nulos por columna

Un nulo en `sucursales_provincia` no es lo mismo que un nulo en `precio`. El
primero lo puedo imputar por sucursal; el segundo hace que la fila no sirva.

In [ ]:
df = con.execute(f"SELECT * FROM read_parquet('{PARQUET}') USING SAMPLE 300000 ROWS").df()
(100 * df.isna().mean()).round(2).sort_values(ascending=False).to_frame('% nulos')

### 3. El mismo EAN con descripciones distintas

Cada cadena escribe el nombre a su manera. Esto es exactamente el motivo por el
que la canasta se arma con patrones de texto y no con EAN puro.

In [ ]:
con.execute(f"""
    SELECT ean, count(DISTINCT descripcion) AS n_desc,
           string_agg(DISTINCT descripcion, ' || ') AS variantes
    FROM read_parquet('{PARQUET}')
    GROUP BY 1 HAVING count(DISTINCT descripcion) > 1
    ORDER BY n_desc DESC LIMIT 15
""").df()

### 4. Estabilidad temporal: saltos de precio sospechosos

Un salto de +200% de un día para el otro casi nunca es inflación: es un error
de carga o un cambio de presentación del producto.

In [ ]:
salt = con.execute(f"""
    WITH d AS (
        SELECT fecha, cadena, ean, median(precio) AS p
        FROM read_parquet('{PARQUET}') GROUP BY 1,2,3
    ), v AS (
        SELECT *, lag(p) OVER (PARTITION BY cadena, ean ORDER BY fecha) AS p_prev FROM d
    )
    SELECT fecha, cadena, ean, p_prev, p,
           round(100.0*(p/nullif(p_prev,0)-1),1) AS var_pct
    FROM v WHERE p_prev IS NOT NULL AND abs(p/nullif(p_prev,0)-1) > 0.5
    ORDER BY abs(var_pct) DESC LIMIT 20
""").df()
display(salt)
print(f'Saltos > 50% diario: {len(salt)}')

### 5. Sucursales fantasma

Sucursales que reportan algunos días y otros no. Si no las trato, el índice se
mueve por composición de la muestra y no por precios.

In [ ]:
con.execute(f"""
    WITH s AS (
        SELECT cadena, id_sucursal, count(DISTINCT fecha) AS dias
        FROM read_parquet('{PARQUET}') GROUP BY 1,2
    )
    SELECT dias, count(*) AS sucursales
    FROM s GROUP BY 1 ORDER BY dias
""").df()

### Decisiones tomadas (para el README)

| Regla | Motivo | % descartado |
|---|---|---|
| Precio nulo o EAN vacío | fila inservible | *completar* |
| Precio fuera de [1, 5.000.000] | error de carga | *completar* |
| Duplicado (comercio, sucursal, EAN) | doble reporte | *completar* |
| Precio > 10x mediana del EAN ese día | coma decimal mal cargada | *completar* |
| Provincias con < 3 cadenas | no son comparables | *completar* |
